# Lab 2: Classical Forecasts for JPM (Naive, Drift, Exponential Smoothing, ARIMA, ARMAX)

**Owner:** Member 2 · **Marking items covered:** 2. Honest validation, 3. Evaluation against the naive forecast, 4. Modelling quality

**The question this notebook answers:** do classical time-series models forecast JPM's next-day log return better than the naive forecast ("tomorrow's price = today's price"), on days that they never saw?

> **How to use this starter (Member 2, delete this box when you finish).** Each step below has a
> markdown cell saying what to do and which part of the course `3_classical_forecasting_baselines.ipynb`
> to copy from. Replace every `# TODO` cell with your code, and every *italic* placeholder with your
> own words about **JPM's** results. Steps 1 and 2 already work: run them first and check that you
> get 2,789 rows and 29 folds.

## Step 1: Set up

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.api import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

from tslab import config
from tslab.data import load_market_data
from tslab.team import make_dataset_jpm                 # OUR features (includes the financial sector), not tslab.features
from tslab.validation import walk_forward_splits, describe_splits
from tslab.metrics import compare_models, diebold_mariano, dm_verdict, directional_accuracy_test
from tslab.plots import plot_walk_forward_layout, plot_price_forecasts

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
np.random.seed(config.SEED)
print("Target stock:", config.TARGET_TICKER)

## Step 2: The data and the walk-forward folds

Every notebook in our project uses exactly these lines, so that every model is scored on the same days.

In [ ]:
frame = load_market_data()                                              # one row per JPM trading day
dataset = make_dataset_jpm(frame, horizon=1).dropna(subset=["target_return"])
splits = list(walk_forward_splits(len(dataset), config.INITIAL_TRAIN, config.TEST_SIZE))

print("Rows in the modelling table:", len(dataset))                     # expect 2789
print("Walk-forward folds         :", len(splits))                      # expect 29
print("First test day:", dataset.index[splits[0][1][0]].date(), "| last test day:", dataset.index[splits[-1][1][-1]].date())
describe_splits(splits, dataset.index).head()

In [ ]:
plot_walk_forward_layout(splits, len(dataset))

### Why this test is honest

*TODO (your words, three bullet points): the training window only contains days before each test block (an expanding window, no shuffling); every model is refitted inside each fold on the training block only; the ARIMA order is chosen on the first training block only, before any test day.*

## Step 3: Baseline forecasts

Copy from Notebook 3, Step 3 (the cells that create `predictions`, `empty_series`, `log_close`, `position_in_frame`, the drift forecast and the SES forecast).

* **Naive:** predict a log return of 0. `compare_models` adds this row automatically.
* **Drift:** the average return of the training block, recomputed in every fold.
* **Simple exponential smoothing:** alpha re-estimated in every fold.

In [ ]:
predictions = {}

def empty_series():
    return pd.Series(np.nan, index=dataset.index)

log_close = np.log(frame["Close"])
position_in_frame = {date: i for i, date in enumerate(frame.index)}

# TODO: drift forecast (Notebook 3, Step 3.2) -> predictions["Drift"]

In [ ]:
# TODO: simple exponential smoothing (Notebook 3, Step 3.4) -> predictions["Simple exponential smoothing"]

## Step 4: ARIMA

Copy from Notebook 3, Step 4. Choose p and q with the AIC on the **first training block only**.

**What to expect on JPM:** the AIC picks ARIMA(0,1,0), a random walk, which is the same as the naive forecast. Report that as a finding, then also run **ARMA(1,0)** on returns (`walk_forward_arma(use_exog=False)` in Step 5) so we have a classical model that differs from naive.

In [ ]:
# TODO: AIC table on the first training block, show the top 5, choose the order

In [ ]:
# TODO: walk-forward ARIMA with the chosen order -> predictions["ARIMA"]

*TODO (your words): what the chosen order means for JPM.*

## Step 5: ARMA(1,0) and ARMAX with our predictors

Copy from Notebook 3, Step 5, and **add `"FIN"`** (the financial sector fund) to the list of predictors. Keep `.shift(1)`.

In [ ]:
# Daily changes of each predictor market (log returns for prices, difference for the interest rate)
predictor_changes = pd.DataFrame(index=frame.index)
for name in ["SP500", "TECH", "GOLD", "BRENT", "USD", "VIX", "FIN"]:      # FIN added by our group
    predictor_changes[name] = np.log(frame[name]).diff()
predictor_changes["US10Y"] = frame["US10Y"].diff()

# THE SHIFT: the predictors used for day s are the ones from day s-1
exog_all = predictor_changes.shift(1)
stock_return = log_close.diff()

# TODO: copy walk_forward_arma from Notebook 3, then:
# predictions["ARMA(1,0)"] = walk_forward_arma(use_exog=False)
# predictions["ARMAX"]     = walk_forward_arma(use_exog=True)

*TODO (one sentence, your words): why the predictors are shifted by one day.*

## Step 6: Compare every model with the naive forecast

This is the most important output of the notebook. The table must show RMSE, MAE, out-of-sample R² and directional accuracy, with the naive forecast in it. Then the Diebold-Mariano test of each model against naive.

In [ ]:
if not predictions:
    print("No models yet: fill in Steps 3 to 5 first.")   # (Member 2: delete this guard when your models work)
else:
    scores = compare_models(dataset, predictions)      # includes the "Naive (no change)" row
    display(scores.round(4))

In [ ]:
# TODO: DM table against naive (copy the Notebook 4 Step 7 DM cell, which also adds the hit rate and its p-value)

*TODO (your words): which classical model has the lowest error? Is it significantly better than naive (DM p below 0.05)? Is the directional accuracy really above 50%?*

## Step 7: A forecast that looks perfect but is not

Copy Notebook 3, Step 7, and explain in two sentences why it looks perfect and why it is wrong.

In [ ]:
# TODO

## Step 8: Residual diagnostics

Copy Notebook 3, Step 8 (Ljung-Box test on the residuals).

In [ ]:
# TODO

*TODO (one sentence): is there autocorrelation left in the residuals?*

## Step 9: What happens 63 days ahead?

Copy Notebook 3, Step 9. Show how the forecast flattens as the horizon grows.

In [ ]:
# TODO

## Step 10: Save the predictions

Member 6 reads this file. Keep the column names exactly as below.

In [ ]:
if predictions:
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    saved = pd.concat(predictions, axis=1)
    saved.insert(0, "Naive (no change)", np.where(saved.notna().any(axis=1), 0.0, np.nan))
    saved.to_csv(config.OUTPUT_DIR / "predictions_classical.csv")
    print("Saved", saved.dropna().shape[0], "rows and columns", list(saved.columns))
else:
    print("Nothing saved yet: no models.")

## Conclusion

*TODO (your words, about JPM's numbers): which model did best, whether any model beats the naive forecast significantly, and what that means for the portfolio manager. If no model beats naive, say so plainly: it is a valid finding.*